# 🦺 Worker Safety Monitoring System — Colab

التطبيق كامل في دفتر واحد: كشف YOLO، ربط المخالفات باللوائح، تقرير مكتوب + PDF، وسجل CSV.
آخر خلية تعطيك **رابط عام للجمهور**.

**قبل التشغيل:**
1. `Runtime → Change runtime type → T4 GPU`
2. ارفع `best.pt` إلى جذر Google Drive (`MyDrive/best.pt`) — أو اتركه والخلية ٢ تطلب منك ترفعه.
3. *(اختياري)* من 🔑 **Secrets** في الشريط الأيسر أضف `LLM_API_KEY` (مفتاح Gemini من aistudio.google.com/apikey) وفعّل **Notebook access**.
   بدونه كل شيء يشتغل، والتقرير ينقل نص اللوائح بدل الشرح.

**التشغيل:** `Runtime → Run all`، ثم انسخ الرابط `https://….gradio.live` من آخر خلية.

In [25]:
# ===== 1. Install ================================================================
!pip install -q ultralytics "gradio>=4.44" openai reportlab

In [26]:
# ===== 2. GPU, Drive and the model file ==========================================
import os
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == 0 else
      "⚠️ none — Runtime -> Change runtime type -> T4 GPU (CPU works, just slower)")

MODEL_PATH = "/content/drive/MyDrive/WorkerSafety25_training_runs/YOLOv8n-2/weights/best.pt"        # <-- edit if best.pt lives elsewhere
OUTPUT_DIR = "/content/drive/MyDrive/safety_demo"    # CSV log + PDF reports (kept on Drive)

from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists(MODEL_PATH):
    print(f"best.pt not found at {MODEL_PATH}. Upload it here instead:")
    from google.colab import files
    MODEL_PATH = os.path.abspath(next(iter(files.upload())))

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB)")
print(f"Output: {OUTPUT_DIR}")

GPU: NVIDIA A100-SXM4-40GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model: /content/drive/MyDrive/WorkerSafety25_training_runs/YOLOv8n-2/weights/best.pt (6.2 MB)
Output: /content/drive/MyDrive/safety_demo


In [27]:
# ===== 3. LLM settings (optional) ================================================
# Read from Colab Secrets. Only LLM_API_KEY is needed; the other two override the
# defaults (Gemini) if you want another OpenAI-compatible provider.
from google.colab import userdata

for name in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL"):
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value.strip()
    except Exception:  # secret missing or notebook access not granted
        pass

print("LLM:", "✅ key found" if os.environ.get("LLM_API_KEY") else
      "⚠️ no LLM_API_KEY — reports will quote the regulations without an AI narrative")

LLM: ✅ key found


In [28]:
%%writefile /content/safety_rules.md
# Workplace Safety Regulations — PPE Compliance Reference

> NOTE: This is a starter knowledge base written in a generic, OSHA-style format
> for demo purposes. Replace or extend the rules below with the actual
> regulations that apply to your organization or jurisdiction.
> Keep the same structure (### RULE heading + fields) so retrieval stays accurate.

---

### RULE HH-01 — Head Protection Required
**Applies to:** All personnel on active construction or industrial sites
**Violation class:** no_hardhat

Every worker must wear an approved hard hat at all times while inside an active
work zone. This requirement applies regardless of the task being performed, the
duration of presence on site, or the worker's role, including visitors and
supervisory staff.

**Hazard:** Head injury from falling objects, swinging loads, fixed obstacles, or
contact with electrical conductors. Head trauma is among the leading causes of
fatal and permanently disabling workplace injuries.

**Severity:** HIGH

**Required action:** Stop the worker immediately. Do not permit the worker to
resume any activity until an approved hard hat is worn correctly. Record the
incident in the site safety log and notify the responsible supervisor.

**Verification:** Hard hat must be worn with the shell upright, the suspension
properly adjusted, and the chin strap fastened when working at height or in
windy conditions.

---

### RULE HH-02 — Hard Hat Condition and Replacement
**Applies to:** All hard hats in service
**Violation class:** damaged_hardhat

A hard hat that has sustained a significant impact must be removed from service
immediately, even if no damage is visible. Shells showing cracks, deep scratches,
chalking, or discolouration from UV exposure must be replaced.

**Hazard:** A compromised shell can fail on impact without warning, giving the
wearer a false sense of protection.

**Severity:** HIGH

**Required action:** Withdraw the hard hat from service and issue a replacement
before the worker returns to the work zone.

---

### RULE HV-01 — High-Visibility Clothing Required
**Applies to:** All personnel in areas with vehicle or mobile plant movement
**Violation class:** no_safety_vest

Workers must wear a high-visibility safety vest or equivalent garment whenever
they are exposed to moving vehicles, mobile equipment, cranes, or any traffic
management area. This includes flaggers, spotters, and anyone crossing haul roads.

**Hazard:** Struck-by incidents involving vehicles and mobile plant. Poor
conspicuity is a primary contributing factor, especially at dawn, dusk, at night,
and in dust, fog, or rain.

**Severity:** HIGH

**Required action:** Remove the worker from the traffic or plant movement area
until a compliant high-visibility garment is worn. Verify the class of garment
matches the exposure level for the area.

**Verification:** The garment must be fastened closed, must not be obscured by
tool belts or outer layers, and retro-reflective banding must be intact and clean.

---

### RULE HV-02 — High-Visibility Garment Condition
**Applies to:** All high-visibility garments in service
**Violation class:** damaged_safety_vest

Garments that are torn, heavily soiled, faded, or have damaged or missing
retro-reflective tape no longer provide the intended level of conspicuity and
must be replaced.

**Severity:** MEDIUM

**Required action:** Replace the garment before the worker re-enters a vehicle or
plant movement area.

---

### RULE PPE-01 — General PPE Duty
**Applies to:** All personnel, all work zones
**Violation class:** general

Personal protective equipment is the final layer of defence, not the first.
Employers must first attempt to eliminate or engineer out the hazard, then apply
administrative controls, and only then rely on PPE. Where PPE is required, the
employer must provide it at no cost to the worker, ensure correct fit, and
provide training on its use, limitations, care, and replacement.

**Severity:** MEDIUM

**Required action:** Where a PPE violation is observed repeatedly at the same
location, review whether an upstream control (barrier, exclusion zone, task
redesign) should be implemented instead of relying on PPE alone.

---

### RULE PPE-02 — Supervisor Accountability
**Applies to:** Site supervisors and safety officers
**Violation class:** general

Supervisors are responsible for enforcing PPE compliance within their area of
control. A supervisor who observes a violation and does not act is themselves in
breach of site safety policy.

**Severity:** MEDIUM

**Required action:** Log the observation, correct it at the time it is observed,
and escalate repeat offences through the site disciplinary process.

---

### RULE REP-01 — Violation Recording
**Applies to:** All observed safety violations
**Violation class:** general

Every observed violation must be recorded with the date, time, location, the
nature of the violation, and the corrective action taken. Records support trend
analysis, allowing the site to identify recurring problem areas rather than
treating each incident in isolation.

**Severity:** LOW

**Required action:** Ensure the violation is captured in the site safety log
before the end of the shift.

---

### RULE ESC-01 — Escalation Thresholds
**Applies to:** Repeat or high-severity violations
**Violation class:** general

A single HIGH severity violation requires immediate correction. Three or more
HIGH severity violations recorded at the same location within a rolling seven-day
period require a formal site safety review and a written corrective action plan.

**Severity:** HIGH

**Required action:** Trigger a documented safety stand-down and review the
adequacy of existing controls for that location.

Overwriting /content/safety_rules.md


In [29]:
# ===== 5. Detector ===============================================================
import re
import threading
from collections import Counter

import numpy as np
from PIL import Image
from ultralytics import YOLO

model = YOLO(MODEL_PATH)
CLASS_NAMES = model.names
print("Model classes:", CLASS_NAMES)


def normalise(name):
    """'NO-Safety Vest' -> 'nosafetyvest'"""
    return re.sub(r"[\s_\-]+", "", name).lower()


# Confirmed from best.pt:
#   {0: 'person', 1: 'hardhat', 2: 'no hardhat', 3: 'safety vest', 4: 'no safety vest'}
VIOLATION_CLASSES = {normalise(n) for n in ("no hardhat", "no safety vest")}
PERSON_CLASSES = {normalise("person")}
CONFIDENCE_THRESHOLD = 0.35

# Fail loudly here rather than silently reporting "no violations" during the demo.
missing = (VIOLATION_CLASSES | PERSON_CLASSES) - {normalise(n) for n in CLASS_NAMES.values()}
assert not missing, f"Expected classes not in the model: {missing}. Update the sets above."

_predict_lock = threading.Lock()  # YOLO is not thread-safe; the public link can get parallel uploads


def detect(image):
    """Returns (annotated PIL image, total persons, violations, detected class names)."""
    with _predict_lock:
        r = model.predict(image.convert("RGB"), conf=CONFIDENCE_THRESHOLD, device=DEVICE, verbose=False)[0]

    violations, detected = [], []
    for box in r.boxes:
        label = CLASS_NAMES[int(box.cls[0])]
        detected.append(label)
        if normalise(label) in VIOLATION_CLASSES:
            violations.append({
                "type": label,
                "confidence": round(float(box.conf[0]), 3),
                "box": [round(float(v)) for v in box.xyxy[0]],
            })

    # The model often boxes a head ("no hardhat") without boxing the person, so
    # count workers as the largest of: person boxes, head boxes, torso boxes.
    n = Counter(normalise(label) for label in detected)
    persons = max(sum(n[c] for c in PERSON_CLASSES),
                  n["hardhat"] + n["nohardhat"],
                  n["safetyvest"] + n["nosafetyvest"])

    annotated = Image.fromarray(r.plot()[:, :, ::-1])  # BGR -> RGB
    return annotated, persons, violations, sorted(set(detected))


# The first inference on a fresh GPU is slow (CUDA init). Pay that cost now.
detect(Image.fromarray(np.zeros((640, 640, 3), dtype=np.uint8)))
print("✅ Detector ready.")

Model classes: {0: 'person', 1: 'hardhat', 2: 'no hardhat', 3: 'safety vest', 4: 'no safety vest'}
✅ Detector ready.


In [30]:
# ===== 6. Safety advisor (RAG + LLM) =============================================
# Maps each violation class to the site regulation it breaches, then asks an LLM
# to write the findings. Falls back to the raw regulation text if the LLM is
# unavailable, so the report always cites real rules.
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set

DEFAULT_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEFAULT_MODEL = "gemini-2.5-flash"

# Keys are normalised class names (see normalise()).
VIOLATION_RULES: Dict[str, List[str]] = {
    "nohardhat": ["HH-01"],
    "nosafetyvest": ["HV-01"],
}
ALWAYS_INCLUDE: List[str] = ["PPE-01"]

SYSTEM_PROMPT = """You are a workplace safety officer writing the findings section of an inspection report.

You will be given:
  - a list of safety violations detected by an automated vision system
  - the relevant excerpts from the site's safety regulations

Write a concise findings section. For each violation:
  - state which rule was breached, citing its rule ID
  - state the severity
  - state the required corrective action

Rules you must follow:
  - Base every statement on the provided regulation excerpts. Do not invent rule
    IDs, severities, or requirements that are not in the excerpts.
  - If the excerpts do not cover something, say so plainly rather than guessing.
  - Write in plain professional English. No preamble, no closing pleasantries.
  - Keep it under 200 words.
"""

_STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "at", "is",
    "are", "be", "must", "that", "this", "with", "as", "by", "from", "any",
    "all", "not", "it", "its", "which", "when", "where", "their", "they",
}


def _tokenise(text):
    return {w for w in re.findall(r"[a-z]+", text.lower()) if len(w) > 2 and w not in _STOPWORDS}


@dataclass
class Rule:
    rule_id: str
    title: str
    text: str
    tokens: Set[str] = field(default_factory=set, repr=False)


@dataclass
class AdvisorResult:
    narrative: str        # LLM-written findings, or the regulation text fallback
    cited_rules: List[str]
    used_llm: bool


def _parse_rules(markdown_text):
    """One Rule per '### RULE <ID> — <title>' section."""
    rules = []
    for chunk in re.split(r"\n(?=### RULE )", markdown_text):
        chunk = chunk.strip()
        m = re.match(r"### RULE\s+([A-Z]+-\d+)\s*[—-]\s*(.+)", chunk.split("\n", 1)[0])
        if not m:
            continue
        body = chunk.replace("---", "").strip()
        rules.append(Rule(m.group(1).strip(), m.group(2).strip(), body, _tokenise(body)))
    return rules


def _field(rule, name):
    """'**Severity:** HIGH' -> 'HIGH', with the rule file's hard line wraps joined."""
    m = re.search(rf"\*\*{name}:\*\*\s*(.+?)(?=\n\s*\n|\n\*\*|\Z)", rule.text, re.S)
    return " ".join(m.group(1).split()) if m else "not specified"


class SafetyAdvisor:
    def __init__(self, rules_path):
        with open(rules_path, encoding="utf-8") as f:
            self._rules = _parse_rules(f.read())
        assert self._rules, "No rules parsed. Each rule must start with '### RULE <ID> — <title>'."
        self.by_id = {r.rule_id: r for r in self._rules}
        print(f"[SafetyAdvisor] Loaded {len(self._rules)} safety rules.")

    def search(self, query, k=3):
        """Keyword overlap; only used for class names missing from VIOLATION_RULES."""
        q = _tokenise(query)
        if not q:
            return []
        scored = sorted(((len(q & r.tokens) / len(q), r) for r in self._rules), key=lambda p: p[0], reverse=True)
        return [r for score, r in scored[:k] if score > 0]

    def rules_for_violations(self, violations):
        rule_ids = []
        for vtype in sorted({v["type"] for v in violations}):
            mapped = VIOLATION_RULES.get(normalise(vtype))
            if mapped is None:
                print(f"[SafetyAdvisor] Unmapped violation class '{vtype}'. Add it to VIOLATION_RULES.")
                mapped = [r.rule_id for r in self.search(vtype.replace("_", " "), k=1)]
            rule_ids.extend(mapped)
        rule_ids.extend(ALWAYS_INCLUDE)
        return [self.by_id[rid] for rid in dict.fromkeys(rule_ids) if rid in self.by_id]

    def explain(self, violations):
        if not violations:
            return AdvisorResult("No safety violations were detected in this image.", [], False)

        rules = self.rules_for_violations(violations)
        rule_ids = [r.rule_id for r in rules]
        violation_list = "\n".join(f"- {t}" for t in sorted({v["type"] for v in violations}))
        narrative = self._ask_llm(violation_list, "\n\n".join(r.text for r in rules))

        if narrative is None:
            fallback = "The following regulations apply to the detected violations:\n\n" + "\n".join(
                f"- **[{r.rule_id}] {r.title}** — Severity: {_field(r, 'Severity')}. "
                f"Required action: {_field(r, 'Required action')}"
                for r in rules
            )
            return AdvisorResult(fallback, rule_ids, used_llm=False)
        return AdvisorResult(narrative, rule_ids, used_llm=True)

    def _ask_llm(self, violation_list, excerpts):
        """Returns None on any failure so the caller can fall back."""
        api_key = os.getenv("LLM_API_KEY", "")
        if not api_key:
            return None
        try:
            from openai import OpenAI

            client = OpenAI(api_key=api_key, base_url=os.getenv("LLM_BASE_URL", DEFAULT_BASE_URL))
            response = client.chat.completions.create(
                model=os.getenv("LLM_MODEL", DEFAULT_MODEL),
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Detected violations:\n{violation_list}\n\n"
                                                f"Relevant safety regulations:\n{excerpts}"},
                ],
                temperature=0.2,
                # Gemini 2.5 "thinks" before answering and those tokens count
                # against this limit; too low a value returns an empty answer.
                max_tokens=2048,
                timeout=30,
            )
            content = response.choices[0].message.content
            return content.strip() if content and content.strip() else None
        except Exception as exc:  # deliberately broad: the report must still generate
            print(f"[SafetyAdvisor] LLM call failed, falling back: {exc}")
            return None


advisor = SafetyAdvisor("/content/safety_rules.md")

if os.environ.get("LLM_API_KEY"):
    test = advisor._ask_llm("- no hardhat", advisor.by_id["HH-01"].text)
    print("LLM test:", "✅ working" if test else "❌ failed (see error above) — reports will use regulation text")

[SafetyAdvisor] Loaded 8 safety rules.
[SafetyAdvisor] LLM call failed, falling back: Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}]
LLM test: ❌ failed (see error above) — reports will use regulation text


In [31]:
# ===== 7. Report (text + PDF) and CSV log ========================================
import csv
import html
from datetime import datetime

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import Image as RLImage
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

LOG_PATH = os.path.join(OUTPUT_DIR, "violations_log.csv")
_log_lock = threading.Lock()


def log_violations(violations, report_id, timestamp):
    if not violations:
        return
    with _log_lock:
        new_file = not os.path.exists(LOG_PATH)
        with open(LOG_PATH, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if new_file:
                writer.writerow(["timestamp", "report_id", "violation", "confidence"])
            for v in violations:
                writer.writerow([timestamp.isoformat(timespec="seconds"), report_id, v["type"], v["confidence"]])


def _summary_line(persons, violations, detected):
    if not detected:
        return "No workers detected in the image."
    if violations:
        names = ", ".join(sorted({v["type"] for v in violations}))
        return f"⚠️ Violations detected: {names} — workers detected: {persons}"
    return f"✅ All {persons} detected worker(s) are compliant with safety standards."


def _references(advice):
    return [f"{rid} — {advisor.by_id[rid].title}" for rid in advice.cited_rules]


def build_report_text(report_id, timestamp, persons, violations, detected, advice):
    """The written report shown on screen. Same content as the PDF."""
    lines = [
        "## Safety Inspection Report",
        f"**Report ID:** {report_id}  ",
        f"**Date:** {timestamp:%Y-%m-%d %H:%M}  ",
        f"**Workers detected:** {persons}  ",
        f"**Violations:** {len(violations)}",
        "",
        f"**Result:** {_summary_line(persons, violations, detected)}",
    ]
    if violations:
        lines += ["", "| # | Violation |", "|---|---|"]
        lines += [f"| {i} | {v['type']} |" for i, v in enumerate(violations, 1)]
    lines += ["", "### Findings", advice.narrative]
    if advice.cited_rules:
        lines += ["", "### References"] + [f"- {ref}" for ref in _references(advice)]
    if violations:
        lines += ["", "*Findings written by AI from the site regulations.*" if advice.used_llm
                  else "*AI narrative unavailable — regulation text quoted directly.*"]
    return "\n".join(lines)


def _pdf_paragraphs(markdown_text, style, bullet_style):
    """Minimal Markdown -> reportlab: **bold**, '- ' bullets, headings as bold lines."""
    out = []
    for block in re.split(r"\n\s*\n", markdown_text.strip()):
        for line in block.split("\n"):
            line = line.strip()
            if not line:
                continue
            is_bullet = bool(re.match(r"^[-*•]\s+", line))
            line = re.sub(r"^[-*•]\s+", "", line)
            line = re.sub(r"^#+\s*", "", line)
            text = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", html.escape(line, quote=False))
            out.append(Paragraph(("• " + text) if is_bullet else text, bullet_style if is_bullet else style))
        out.append(Spacer(1, 0.2 * cm))
    return out


def generate_pdf_report(report_id, timestamp, annotated, persons, violations, detected, advice):
    pdf_path = os.path.join(OUTPUT_DIR, f"{report_id}.pdf")
    image_path = os.path.join(OUTPUT_DIR, f"{report_id}.jpg")
    annotated.save(image_path, format="JPEG", quality=88)

    styles = getSampleStyleSheet()
    body = styles["BodyText"]
    bullet = styles["BodyText"].clone("bullet", leftIndent=12)

    w, h = annotated.size
    img_w = 16 * cm
    img_h = img_w * h / w
    if img_h > 11 * cm:
        img_h, img_w = 11 * cm, 11 * cm * w / h

    story = [
        Paragraph("Safety Inspection Report", styles["Title"]),
        Paragraph(f"Report ID: {report_id} &nbsp;&nbsp; Date: {timestamp:%Y-%m-%d %H:%M}", body),
        Spacer(1, 0.4 * cm),
        RLImage(image_path, width=img_w, height=img_h),
        Spacer(1, 0.4 * cm),
        Paragraph(f"Workers detected: <b>{persons}</b> &nbsp;&nbsp; Violations: <b>{len(violations)}</b>", body),
        Paragraph(html.escape(_summary_line(persons, violations, detected).lstrip("⚠️✅ ")), body),
    ]

    if violations:
        rows = [["#", "Violation"]] + [[str(i), v["type"]] for i, v in enumerate(violations, 1)]
        table = Table(rows, colWidths=[1.2 * cm, 13.5 * cm])
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0f766e")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#cbd5e1")),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f1f5f9")]),
        ]))
        story += [Spacer(1, 0.3 * cm), table]

    story += [Spacer(1, 0.5 * cm), Paragraph("Findings", styles["Heading2"])]
    story += _pdf_paragraphs(advice.narrative, body, bullet)

    if advice.cited_rules:
        story.append(Paragraph("References", styles["Heading3"]))
        story += [Paragraph("• " + html.escape(ref), bullet) for ref in _references(advice)]
    if violations:
        note = ("Findings written by AI from the site regulations." if advice.used_llm
                else "AI narrative unavailable — regulation text quoted directly.")
        story += [Spacer(1, 0.3 * cm), Paragraph(f"<i>{note}</i>", body)]

    SimpleDocTemplate(pdf_path, pagesize=A4, topMargin=1.5 * cm, bottomMargin=1.5 * cm,
                      title=f"Safety Inspection Report {report_id}").build(story)
    return pdf_path


print("Report + log ready. Log file:", LOG_PATH)

Report + log ready. Log file: /content/drive/MyDrive/safety_demo/violations_log.csv


In [32]:
# ===== 8. Web app with a public link =============================================
import uuid
from collections import Counter

import gradio as gr

CSS = """
h1 { font-size: 2.6rem; font-weight: 800; margin: 16px 0 4px; }
.card { border-left: 4px solid var(--color-accent); border-radius: 8px; padding: 16px 20px;
        background: var(--block-background-fill); }
.card p { margin: 6px 0 0; color: var(--body-text-color-subdued); }
.big { font-size: 2.2rem; font-weight: 800; color: var(--color-accent); }
.alert { border-left-color: #f59e0b; }
.alert .big { color: #f59e0b; }
.name { font-size: 1.1rem; font-weight: 700; color: var(--body-text-color); }
"""


def verdict(persons, violations, detected):
    if not detected:
        return '<div class="card"><div class="big">No workers found</div></div>'
    if not violations:
        return f'<div class="card"><div class="big">Compliant</div><p>{persons} worker(s) checked</p></div>'
    rows = ""
    for vtype, n in Counter(v["type"] for v in violations).items():
        rule = advisor.by_id.get((VIOLATION_RULES.get(normalise(vtype)) or [""])[0])
        ref = f"{rule.rule_id} {rule.title} · {_field(rule, 'Severity')}" if rule else ""
        rows += f'<p><span class="name">{vtype.capitalize()} × {n}</span> — {ref}</p>'
    return f'<div class="card alert"><div class="big">{len(violations)} violations</div>{rows}</div>'


def process_image(image):
    if image is None:
        return None, "", "", None
    ts = datetime.now()
    rid = f"{ts:%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:4]}"
    annotated, persons, violations, detected = detect(image)
    log_violations(violations, rid, ts)
    advice = advisor.explain(violations)
    report = build_report_text(rid, ts, persons, violations, detected, advice).replace("⚠️ ", "").replace("✅ ", "")
    pdf = generate_pdf_report(rid, ts, annotated, persons, violations, detected, advice)
    return annotated, verdict(persons, violations, detected), report, pdf


with gr.Blocks(title="Worker Safety Monitoring") as demo:
    gr.HTML("<h1>Worker Safety Monitoring</h1><p>Upload a job-site photo to check hard hats and safety vests.</p>")
    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type="pil", label="Site photo", placeholder="Drop an image here or click to upload")
            button = gr.Button("Analyze image", variant="primary")
        with gr.Column():
            image_out = gr.Image(label="Detections")
            card = gr.HTML()
    report_md = gr.Markdown()
    report_pdf = gr.File(label="PDF report")
    button.click(process_image, image_in, [image_out, card, report_md, report_pdf])

if globals().get("running_demo") is not None:
    running_demo.close()
running_demo = demo
demo.launch(share=True, allowed_paths=[OUTPUT_DIR], footer_links=[], css=CSS,
            theme=gr.themes.Soft(primary_hue="teal", neutral_hue="slate"))

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3df7f697fbb93de91.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
